# Tech Challenge IADT Fase 4 - Sistema Multimodal Completo

# Apresentacao: 15 minutos

## Objetivo

Desenvolver um sistema de analise multimodal para monitoramento medico, integrando analise de video, audio e detecao de anomalias

In [ ]:
import cv2
import numpy as np
import torch
import mediapipe as mp
import pandas as pd
from scipy import stats
import os

# ---- Configuracao de execucao (switch em tempo de execucao) ----
# MODO = "offline"  -> sem Azure: usa transcricao/texto e frames simulados
# MODO = "azure"    -> usa Azure Cognitive Services (exige chaves e pacotes)
try:
    MODO = input("Modo de execucao [offline/azure] (Enter = offline): ").strip().lower() or "offline"
except Exception:
    MODO = os.getenv("MODO", "offline")
CAMINHO_VIDEO = os.getenv("CAMINHO_VIDEO", "dados/video.mp4")
CAMINHO_AUDIO = os.getenv("CAMINHO_AUDIO", "dados/audio.wav")
AZURE_ON = (MODO == "azure")
print("MODO de execucao:", MODO)

# ---- MediaPipe Pose (OpenPose-Light) ----
try:
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    _mp_ok = True
except Exception:
    pose = None
    _mp_ok = False

In [ ]:
try:
    model_yolo = torch.hub.load('ultralytics/yolov8', 'yolov8n', pretrained=True)
except Exception:
    model_yolo = None
_yolo_online = model_yolo is not None
print("YOLO disponivel:", _yolo_online)

In [ ]:
def detectar_objetos_yolo(frame):
    if _yolo_online and model_yolo is not None:
        results = model_yolo(frame)
        objetos = []
        for detection in results.xyxy[0]:
            x1, y1, x2, y2, conf, cls = detection
            label = model_yolo.names[int(cls)]
            obj = {'label': label, 'confidence': float(conf)}
            obj['bbox'] = [int(x1), int(y1), int(x2), int(y2)]
            objetos.append(obj)
        return objetos
    else:
        # Modo offline: objetos simulados para demostracao
        obj1 = {'label': 'instrumento_critico', 'confidence': 0.85, 'bbox': [100, 100, 200, 300]}
        obj2 = {'label': 'area_perigosa', 'confidence': 0.72, 'bbox': [300, 50, 400, 200]}
        return [obj1, obj2]

In [ ]:
def detectar_postura(frame):
    """Detecta landmarks posturais; retorna SEMPRE um dict."""
    if _mp_ok and pose is not None:
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            keypoints = {}
            nomes = ['nariz', 'ldeltoide', 'rdeltoide', 'lquadril', 'rquadril',
                     'ljoelho', 'rjoelho', 'ltobillo', 'rtobillo']
            for name, lm in zip(nomes, landmarks):
                keypoints[name] = (lm.x, lm.y, lm.z)
            return {'status': 'sucesso', 'keypoints': keypoints}
    return {'status': 'nenhum_corpo_detectado'}

In [ ]:
def processar_video_clinico(caminho_video, max_frames=50):
    cap = cv2.VideoCapture(caminho_video)
    resultados = []
    frame_count = 0
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        objetos = detectar_objetos_yolo(frame)
        postura = detectar_postura(frame)
        score_anomalia = 0
        alertas = []
        if postura['status'] == 'sucesso':
            kpts = postura['keypoints']
            lje = kpts.get('ljoelho', (0,0,0))
            rje = kpts.get('rjoelho', (0,0,0))
            if lje[2] > 0.8 or rje[2] > 0.8:
                score_anomalia += 1
                alertas.append('Joelho esticada')
            if lje[2] < 0.2 or rje[2] < 0.2:
                score_anomalia += 1
                alertas.append('Joelho dobrado excessivamente')
        if len(objetos) > 0:
            labels_criticos = [o['label'] for o in objetos if o['confidence'] > 0.5]
            alertas.append('Objetos detectados: %s' % ', '.join(labels_criticos[:3]))
            score_anomalia += len(labels_criticos)
        resultados.append({'frame': frame_count, 'objetos': len(objetos), 'postura_ok': postura['status'] == 'sucesso', 'score_anomalia': score_anomalia, 'alertas': '; '.join(alertas) if alertas else 'Nenhum'})
        frame_count += 1
    cap.release()
    return resultados

## 2. Analise de Audio Medico (5 min)

In [ ]:
try:
    import azure.cognitiveservices.speech as speechsdk
    from azure.ai.textanalytics import TextAnalyticsClient
    from azure.core.credentials import AzureKeyCredential
    _HAS_AZURE_LIBS = True
except Exception:
    _HAS_AZURE_LIBS = False

speech_key = os.getenv("AZURE_SPEECH_KEY", "")
speech_region = os.getenv("AZURE_SPEECH_REGION", "westus2")
text_key = os.getenv("AZURE_TEXT_ANALYTICS_KEY", "")
text_region = os.getenv("AZURE_TEXT_ANALYTICS_REGION", "westus2")

_azure_on = bool(AZURE_ON and _HAS_AZURE_LIBS and speech_key and text_key)
if _azure_on:
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=speech_region)
    speech_config.speech_recognition_language = "pt-BR"
    text_analytics_client = TextAnalyticsClient(
        endpoint=f"https://{text_region}.api.cognitive.microsoft.com/",
        credential=AzureKeyCredential(text_key),
    )
else:
    speech_config = None
    text_analytics_client = None

if AZURE_ON and not _azure_on:
    print("MODO=azure pedido, mas sem pacotes/chaves -> caindo em OFFLINE")
print("Azure configurado" if _azure_on else "Modo OFFLINE (sem Azure)")

In [ ]:
def transcrever_audio(caminho_audio):
    """Transcreve audio com Azure. Offline/sem arquivo -> string vazia."""
    if not _azure_on:
        return ""
    try:
        audio_config = speechsdk.AudioConfig(filename=caminho_audio)
        speech_recognizer = speechsdk.SpeechRecognizer(
            speech_config=speech_config, audio_config=audio_config)
        result = speech_recognizer.recognize_once()
        if result.reason == speechsdk.ResultReason.RecognizedSpeech:
            return result.text
        return ""
    except Exception as e:
        print(f"Erro na transcricao: {e}")
        return ""

In [ ]:
def analisar_texto_azure(texto):
    """Sentimento + entidades. Offline usa heuristica simples; Azure usa Text Analytics."""
    padrao = {'sentimento': {'rotulo': 'neutro',
                             'pontuacoes': {'positivo': 0.0, 'negativo': 0.0, 'neutro': 1.0}},
              'entidades': []}
    if not texto or len(texto.strip()) < 3:
        return padrao
    if not _azure_on:
        t = texto.lower()
        negativos = ['dor', 'cansado', 'fadiga', 'falta de ar', 'exausto',
                     'complicacao', 'desconforto', 'socorro']
        neg = any(w in t for w in negativos)
        return {'sentimento': {'rotulo': ('negativo' if neg else 'neutro'),
                               'pontuacoes': {'positivo': 0.0,
                                              'negativo': (1.0 if neg else 0.0),
                                              'neutro': (0.0 if neg else 1.0)}},
                'entidades': []}
    try:
        sentiment_result = text_analytics_client.analyze_sentiment([texto])[0]
        entity_result = text_analytics_client.recognize_entities([texto])[0]
    except Exception as e:
        print(f"Azure Text Analytics nao disponivel: {e}")
        return padrao
    entidades_medicas = []
    for entity in entity_result.entities:
        if entity.category in ['Person', 'Organization', 'Quantity', 'Temperature', 'DateTime']:
            entidades_medicas.append({'texto': entity.text, 'tipo': entity.category,
                                      'confianca': entity.confidence_score})
    cs = sentiment_result.confidence_scores
    return {'sentimento': {'rotulo': sentiment_result.sentiment,
                           'pontuacoes': {'positivo': float(cs.positive),
                                          'negativo': float(cs.negative),
                                          'neutro': float(cs.neutral)}},
            'entidades': entidades_medicas}

In [ ]:
def detectar_alteracoes_vocais(texto_transcrito):
    padroes_clinicos = {'fadiga': ['cansado', 'fadiga', 'sono', 'exausto'], 'dificuldade_respiratoria': ['dificuldade para respirar', 'falta de ar'], 'dor': ['dor', 'complicacao']}
    texto_lower = texto_transcrito.lower() if texto_transcrito else ''
    resultados = {}
    for condicao, palavras in padroes_clinicos.items():
        encontradas = [p for p in palavras if p in texto_lower]
        resultados[condicao] = {'presente': len(encontradas) > 0, 'encontradas': encontradas, 'quantidade': len(encontradas)}
    return resultados

In [ ]:
def detectar_anomalias_sinais_vitais(df_sinais):
    anomalia = {'fc_anomalia': False, 'bp_anomalia': False, 'spo2_anomalia': False, 'alertas': []}
    if 'fc' in df_sinais.columns:
        fc = df_sinais['fc']
        fc_media = fc.mean()
        fc_desvio = fc.std()
        fc_limite_superior = fc_media + (3 * fc_desvio)
        fc_limite_inferior = fc_media - (3 * fc_desvio)
        fc_atual = df_sinais['fc'].iloc[-1]
        if fc_atual > fc_limite_superior:
            anomalia['fc_anomalia'] = True
            anomalia['alertas'].append('FC CRITICA: %.1f bpm' % fc_atual)
        elif fc_atual < fc_limite_inferior:
            anomalia['fc_anomalia'] = True
            anomalia['alertas'].append('FC CRITICA: %.1f bpm' % fc_atual)
        else:
            anomalia['alertas'].append('FC normal: %.1f bpm' % fc_atual)
    if 'spo2' in df_sinais.columns:
        spo2 = df_sinais['spo2']
        spo2_atual = spo2.iloc[-1]
        if spo2_atual < 95:
            anomalia['spo2_anomalia'] = True
            anomalia['alertas'].append('SpO2 BAIXA: %.1f%%' % spo2_atual)
        else:
            anomalia['alertas'].append('SpO2 normal: %.1f%%' % spo2_atual)
    return anomalia

In [ ]:
def detectar_anomalias_prescricoes(lista_prescricoes):
    anomalia = {'medicamento_novo': False, 'dosagem_alterada': False, 'interacao_drogas': False, 'alertas': []}
    if len(lista_prescricoes) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    atual = lista_prescricoes[-1]
    anterior = lista_prescricoes[-2]
    keys_atual = set(k.lower() for k in atual.keys())
    keys_anterior = set(k.lower() for k in anterior.keys())
    novos = keys_atual - keys_anterior
    if novos:
        anomalia['medicamento_novo'] = True
        anomalia['alertas'].append('Novo medicamento: %s' % ', '.join([k for k in atual.keys() if k.lower() in novos]))
    comuns = keys_atual & keys_anterior
    for med in comuns:
        if atual.get(med) != anterior.get(med):
            anomalia['dosagem_alterada'] = True
            anomalia['alertas'].append('Dosagem alterada para: %s' % med)
    if anomalia['medicamento_novo'] and anomalia['dosagem_alterada']:
        anomalia['interacao_drogas'] = True
        anomalia['alertas'].append('WARNING: Possivel interacao: novo medicamento + cambio de dosagem')
    return anomalia

In [ ]:
def detectar_anomalias_movimentacao(keypoints_history):
    anomalia = {'movimento_repetino': False, 'imobilidade_anormal': False, 'quedas_suspeitas': False, 'alertas': []}
    if len(keypoints_history) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    centoidos = []
    for kp_data in keypoints_history:
        lq = kp_data.get('lquadril', (0, 0))
        rq = kp_data.get('rquadril', (0, 0))
        centroide = ((lq[0] + rq[0]) / 2, (lq[1] + rq[1]) / 2)
        centoidos.append(centroide)
    if len(centoidos) >= 2:
        velocidades = []
        for i in range(1, len(centoidos)):
            dist = ((centoidos[i][0] - centoidos[i-1][0])**2 + (centoidos[i][1] - centoidos[i-1][1])**2)**0.5
            velocidades.append(dist)
        if velocidades:
            velocidade_max = max(velocidades)
            velocidade_media = sum(velocidades) / len(velocidades)
            if velocidade_max > 3 * velocidade_media and velocidade_media > 0:
                anomalia['movimento_repetino'] = True
                anomalia['alertas'].append('Movimento repentino: dist %.3f' % velocidade_max)
            N = min(10, len(velocidades))
            ultimas = velocidades[-N:]
            vm = sum(ultimas) / len(ultimas) if ultimas else 0
            if vm < 0.05 and len(ultimas) >= 10:
                anomalia['imobilidade_anormal'] = True
                anomalia['alertas'].append('Imobilidade: %d frames sem movimento' % len([v for v in ultimas if v < 0.01]))
            if anomalia['movimento_repetino'] and anomalia['imobilidade_anormal']:
                anomalia['quedas_suspeitas'] = True
                anomalia['alertas'].append('WARNING: Padrao de queda suspeito: movimento repentino + imobilidade')
    return anomalia

In [ ]:
def workflow_integrado(resultado_video, resultado_audio, resultado_sinais):
    alertas_gerais = []
    punteis_risco = 0
    if resultado_video and 'total_alertas' in resultado_video:
        vs = resultado_video.get('alertas_listas', [])
        sig = [a for a in vs if 'Nenhum' not in a]
        if sig:
            alertas_gerais.extend(sig)
            punteis_risco += len(sig) * 2
    if resultado_audio and 'total_alteracoes' in resultado_audio:
        aps = [k for k, v in resultado_audio['alteracoes'].items() if v.get('presente', False)]
        if aps:
            mapa = {'fadiga': 'Paciente relata fadiga', 'dificuldade_respiratoria': 'Paciente relata dificuldade respiratoria', 'dor': 'Paciente relata dor', 'alerta_urgencia': 'ALERTA URGENCIA: solicitacao de ajuda'}
            for a in aps:
                m = mapa.get(a, a)
                alertas_gerais.append(m)
                punteis_risco += 3
    if resultado_sinais and 'alertas' in resultado_sinais:
        alertas_gerais.extend(resultado_sinais['alertas'])
        punteis_risco += 3
    if punteis_risco >= 6:
        decisao = 'ALERTA MÉDICO IMEDIATO'
        nivel = 'CRITICO'
        acao = 'Contatar equipe médica imediatamente'
    elif punteis_risco >= 3:
        decisao = 'ALERTA - Revisao necessaria'
        nivel = 'ALTO'
        acao = 'Avaliar paciente e monitorar 30 min'
    else:
        decisao = 'Monitoramento normal continuo'
        nivel = 'Normal'
        acao = 'Continuar rotina de cuidados'
    return {'decisao': decisao, 'nivel': nivel, 'acao_recomendada': acao, 'total_alertas': len(alertas_gerais), 'alertas': '; '.join(alertas_gerais) if alertas_gerais else 'Nenhum motivo de alerta', 'punteis_risco': punteis_risco}

## 4. Execução do Pipeline Multimodal (vídeo + áudio + anomalias)
Gera `resultado_video`, `resultado_audio` e `resultado_sinais` que alimentam o relatório/decisão final.

In [ ]:
# ===== EXECUCAO: ANALISE DE VÍDEO =====
def gerar_frames_sinteticos(n=40):
    """Frames simulados para rodar YOLO/MediaPipe sem arquivo real."""
    frames = []
    for i in range(n):
        base = np.full((480, 640, 3), 60 + i, dtype=np.uint8)
        frames.append(base)
    return frames

def executar_video():
    resultados = None
    if os.path.exists(CAMINHO_VIDEO):
        try:
            resultados = processar_video_clinico(CAMINHO_VIDEO)
        except Exception as e:
            print("Falha no video real, usando frames sinteticos:", e)
            resultados = None
    if not resultados:
        resultados = []
        for idx, frame in enumerate(gerar_frames_sinteticos()):
            objetos = detectar_objetos_yolo(frame)
            postura = detectar_postura(frame)
            alertas, score = [], 0
            if postura.get('status') == 'sucesso':
                kpts = postura.get('keypoints', {})
                lje = kpts.get('ljoelho', (0, 0, 0)); rje = kpts.get('rjoelho', (0, 0, 0))
                if lje[2] > 0.8 or rje[2] > 0.8:
                    score += 1; alertas.append('Joelho esticado')
                if lje[2] < 0.2 or rje[2] < 0.2:
                    score += 1; alertas.append('Joelho dobrado excessivamente')
            if objetos:
                labels = [o['label'] for o in objetos if o['confidence'] > 0.5]
                if labels:
                    score += len(labels)
                    alertas.append('Objetos detectados: ' + ', '.join(labels[:3]))
            resultados.append({'frame': idx, 'objetos': len(objetos),
                               'postura_ok': postura.get('status') == 'sucesso',
                               'score_anomalia': score,
                               'alertas': '; '.join(alertas) if alertas else 'Nenhum'})
    alertas_listas = [r['alertas'] for r in resultados if r['alertas'] != 'Nenhum']
    return {'frames': len(resultados),
            'objetos_totais': sum(r['objetos'] for r in resultados),
            'anomalias_posturais': sum(r['score_anomalia'] for r in resultados),
            'alertas_listas': alertas_listas,
            'total_alertas': len(alertas_listas)}

resultado_video = executar_video()
print("Video: {} frames | {} alertas".format(resultado_video['frames'],
                                              resultado_video['total_alertas']))

In [ ]:
# ===== EXECUCAO: ANALISE DE ÁUDIO =====
def executar_audio():
    texto = ""
    if os.path.exists(CAMINHO_AUDIO):
        texto = transcrever_audio(CAMINHO_AUDIO)
    if not texto:
        texto = ("Paciente: estou muito cansado e com falta de ar. "
                 "Sinto dor no peito desde ontem.")
    analise = analisar_texto_azure(texto)
    alteracoes = detectar_alteracoes_vocais(texto)
    return {'transcricao': texto,
            'sentimento': analise.get('sentimento', {}),
            'entidades': analise.get('entidades', []),
            'alteracoes': alteracoes,
            'total_alteracoes': sum(1 for v in alteracoes.values() if v.get('presente'))}

resultado_audio = executar_audio()
print("Audio: sentimento={} | {} alteracoes vocais".format(
    resultado_audio['sentimento'].get('rotulo', 'neutro'),
    resultado_audio['total_alteracoes']))

In [ ]:
# ===== EXECUCAO: DETECÇÃO DE ANOMALIAS =====
def executar_sinais():
    df_sinais = pd.DataFrame({
        'fc':   [72, 75, 74, 78, 80, 82, 130],
        'spo2': [98, 98, 97, 96, 96, 95, 92],
    })
    resultado_sinais = detectar_anomalias_sinais_vitais(df_sinais)
    prescricoes = detectar_anomalias_prescricoes([
        {'Dipirona': '500mg'},
        {'Dipirona': '1g', 'Oxigenio': '2L/min'},
    ])
    movimentacao = detectar_anomalias_movimentacao([
        {'lquadril': (0.5, 0.5), 'rquadril': (0.6, 0.5)},
        {'lquadril': (0.9, 0.9), 'rquadril': (1.0, 0.9)},
        {'lquadril': (0.9, 0.9), 'rquadril': (1.0, 0.9)},
    ])
    alertas = list(resultado_sinais.get('alertas', []))
    alertas += prescricoes.get('alertas', []) + movimentacao.get('alertas', [])
    resultado_sinais['prescricoes'] = prescricoes
    resultado_sinais['movimentacao'] = movimentacao
    resultado_sinais['alertas'] = alertas
    return resultado_sinais

resultado_sinais = executar_sinais()
print("Sinais/anomalias ->", resultado_sinais['alertas'])

# Relatório Final para Apresentacao de 15 Minutos

In [ ]:
print('=' * 70)
print('TECH CHALLENGE IADT FASE 4 - SISTEMA MULTIMODAL COMPLETO')
print('Modo de execucao:', MODO.upper())
print('=' * 70)

print('\n## 1. DEMONSTRAÇÃO DE ANÁLISE DE VÍDEO')
print('   • OpenPose-Light/MediaPipe: analise postural em {} frames'.format(resultado_video['frames']))
print('   • YOLOv8: {} objetos/areas criticos detectados'.format(resultado_video['objetos_totais']))
print('   • Anomalias posturais (score): {}'.format(resultado_video['anomalias_posturais']))
print('   • Alertas de video: {}'.format(resultado_video['total_alertas']))

print('\n## 2. DEMONSTRAÇÃO DE ANÁLISE DE ÁUDIO')
print('   • Transcricao: "{}"'.format(resultado_audio['transcricao'][:80]))
print('   • Sentimento: {}'.format(resultado_audio['sentimento'].get('rotulo', 'neutro')))
print('   • Entidades medicas: {}'.format(len(resultado_audio['entidades'])))
print('   • Alteracoes vocais: {}'.format(resultado_audio['total_alteracoes']))

print('\n## 3. DETECÇÃO DE ANOMALIAS')
print('   • Sinais/prescricoes/movimentacao ->', '; '.join(resultado_sinais.get('alertas', ['nenhuma'])))

print('\n## 4. INTEGRAÇÃO E DECISÃO FINAL')
resultado = workflow_integrado(resultado_video, resultado_audio, resultado_sinais)
print('   • Decisao:', resultado['decisao'])
print('   • Nivel:', resultado['nivel'])
print('   • Acao:', resultado['acao_recomendada'])
print('   • Alertas gerados:', resultado['total_alertas'])
print('   • Pontuacao de risco:', resultado['punteis_risco'])
print('   • Detalhe:', resultado['alertas'])

print('\n' + '=' * 70)
print('FIM DA APRESENTAÇÃO DO TECH CHALLENGE IADT FASE 4')
print('=' * 70)